# Feature table analysis

This notebook analyzes `analysis/features.json` as a feature table (enums + notes).

It provides:
- Quick sanity checks
- Crosstabs / slices
- Optional clustering (one-hot features)


In [30]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

# Robustly locate repo root even if notebook runs from /analysis
REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / 'analysis' / 'features.json').exists():
  pass
elif (REPO_ROOT.parent / 'analysis' / 'features.json').exists():
  REPO_ROOT = REPO_ROOT.parent
else:
  raise FileNotFoundError('Could not locate analysis/features.json from current working directory')

FEATURES_PATH = REPO_ROOT / 'analysis' / 'features.json'

with FEATURES_PATH.open('r', encoding='utf-8') as f:
  data = json.load(f)

rows = data.get('rows', [])
print('repo_root', REPO_ROOT)
print('rows', len(rows))
assert rows, 'No rows found. Run scripts/extract_feature_table.py first.'


repo_root /Users/kihyvr/Documents/design-system-analysis
rows 58


In [31]:
# Flatten into a dataframe (schema-driven, fills missing keys with 'unknown')
schema_path = REPO_ROOT / 'analysis' / 'features.schema.json'
schema = json.loads(schema_path.read_text(encoding='utf-8'))
feature_keys = [f['key'] for f in schema.get('features', [])]

records = []
for r in rows:
  company = r.get('company', {}) or {}
  features = (r.get('features', {}) or {}).copy()
  for k in feature_keys:
    features.setdefault(k, 'unknown')
  records.append({
    'slug': company.get('slug', ''),
    'name': company.get('name', ''),
    'notes': r.get('notes', ''),
    **{k: features.get(k, 'unknown') for k in feature_keys}
  })

df = pd.DataFrame.from_records(records).sort_values('slug').reset_index(drop=True)
feature_cols = feature_keys

df.head(3)


,slug,name,notes,productType,collectionBucket,uxMode,imageryUsage,colorStrategy,typographyTone,layoutDensity,surfaceDepth,themeMode,shapeLanguage,shadowStyle,contentFocus,motionUsage,primaryIntent
0,airbnb,Airbnb,"Airbnb's design exudes a warm, inviting atmosp...",marketplace,enterpriseAndConsumer,browsingHeavy,imageFirst,singleAccent,expressive,balanced,layered,lightFirst,rounded,stacked,photography,medium,exploration
1,airtable,Airtable,Airtable's design exudes a vibe of sophisticat...,digital,designAndProductivity,browsingHeavy,mixed,multiAccent,neutral,balanced,layered,lightFirst,rounded,subtle,productScreenshots,medium,trust
2,apple,Apple,"Apple's design exudes a premium, cinematic vib...",physical,enterpriseAndConsumer,browsingHeavy,imageFirst,singleAccent,premium,sparse,flat,darkFirst,pill,subtle,photography,low,trust


In [32]:
# Sanity checks
print('unique slugs:', df['slug'].nunique())
print('missing slugs:', (df['slug'] == '').sum())

for c in feature_cols:
  print(c, df[c].value_counts(dropna=False).to_dict())


unique slugs: 58
missing slugs: 0
productType {'digital': 51, 'physical': 6, 'marketplace': 1}
collectionBucket {'developerToolsAndPlatforms': 14, 'aiAndMachineLearning': 12, 'designAndProductivity': 10, 'enterpriseAndConsumer': 7, 'infrastructureAndCloud': 6, 'automotiveAndMobility': 5, 'fintechAndCrypto': 4}
uxMode {'browsingHeavy': 25, 'mixed': 14, 'taskFocused': 12, 'creationTool': 6, 'docsHeavy': 1}
imageryUsage {'mixed': 44, 'imageFirst': 13, 'textFirst': 1}
colorStrategy {'multiAccent': 22, 'singleAccent': 17, 'monochrome': 12, 'gradientLed': 7}
typographyTone {'technical': 25, 'expressive': 17, 'premium': 9, 'neutral': 7}
layoutDensity {'balanced': 34, 'dense': 17, 'sparse': 7}
surfaceDepth {'flat': 30, 'layered': 22, 'subtle': 6}
themeMode {'lightFirst': 28, 'darkFirst': 21, 'dual': 9}
shapeLanguage {'pill': 21, 'rounded': 13, 'sharp': 13, 'mixed': 11}
shadowStyle {'subtle': 20, 'none': 20, 'stacked': 13, 'ring': 5}
contentFocus {'productScreenshots': 13, 'mixed': 13, 'codeFir

In [33]:
# Crosstab helpers

def crosstab(a: str, b: str | None = None):
  if b:
    t = pd.crosstab(df[a], df[b])
  else:
    t = df[a].value_counts().to_frame('count')
  return t

crosstab('collectionBucket')


,count
collectionBucket,
developerToolsAndPlatforms,14
aiAndMachineLearning,12
designAndProductivity,10
enterpriseAndConsumer,7
infrastructureAndCloud,6
automotiveAndMobility,5
fintechAndCrypto,4


In [34]:
# Example: compare businessModel vs imageryUsage
crosstab('collectionBucket', 'imageryUsage')


imageryUsage,imageFirst,mixed,textFirst
collectionBucket,,,
aiAndMachineLearning,1,10,1
automotiveAndMobility,5,0,0
designAndProductivity,2,8,0
developerToolsAndPlatforms,1,13,0
enterpriseAndConsumer,4,3,0
fintechAndCrypto,0,4,0
infrastructureAndCloud,0,6,0


In [35]:
# Slice helper: get examples with notes

def examples(where: dict[str, str], limit: int = 10):
  q = df
  for k, v in where.items():
    q = q[q[k] == v]
  cols = ['slug', 'name', 'notes'] + [k for k in where.keys()]
  return q[cols].head(limit)

examples({'collectionBucket': 'devtools', 'colorStrategy': 'monochrome'}, limit=10)


,slug,name,notes,collectionBucket,colorStrategy


In [36]:
# Optional clustering (one-hot encode enum features)
# If scikit-learn isn't installed, install it in your environment:
#   pip install scikit-learn

from sklearn.cluster import KMeans

X = pd.get_dummies(df[feature_cols], prefix=feature_cols)

k = 5
km = KMeans(n_clusters=k, n_init='auto', random_state=7)
labels = km.fit_predict(X)

df_clustered = df.assign(cluster=labels)
df_clustered['cluster'].value_counts().sort_index()


cluster
0    13
1    14
2    14
3    10
4     7
Name: count, dtype: int64

In [37]:
# Inspect clusters: most common feature values per cluster

def cluster_profile(cluster_id: int):
  sub = df_clustered[df_clustered['cluster'] == cluster_id]
  out = {}
  for c in feature_cols:
    out[c] = sub[c].value_counts().head(3).to_dict()
  return pd.Series(out)

cluster_profile(0)


productType                                           {'digital': 13}
collectionBucket    {'developerToolsAndPlatforms': 6, 'designAndPr...
uxMode              {'browsingHeavy': 5, 'taskFocused': 4, 'creati...
imageryUsage                           {'mixed': 12, 'imageFirst': 1}
colorStrategy       {'monochrome': 4, 'multiAccent': 3, 'singleAcc...
typographyTone           {'technical': 5, 'neutral': 3, 'premium': 3}
layoutDensity                           {'balanced': 10, 'sparse': 3}
surfaceDepth                   {'flat': 7, 'layered': 4, 'subtle': 2}
themeMode                                          {'lightFirst': 13}
shapeLanguage                   {'pill': 6, 'rounded': 4, 'mixed': 3}
shadowStyle                    {'subtle': 8, 'none': 3, 'stacked': 2}
contentFocus        {'productScreenshots': 9, 'mixed': 2, 'illustr...
motionUsage                                   {'low': 9, 'medium': 4}
primaryIntent                         {'trust': 12, 'exploration': 1}
dtype: object

In [38]:
# Save a CSV snapshot for ad-hoc exploration
OUT_CSV = REPO_ROOT / 'analysis' / 'features.csv'
df.to_csv(OUT_CSV, index=False)
print('Wrote', OUT_CSV)


Wrote /Users/kihyvr/Documents/design-system-analysis/analysis/features.csv


## Clustering & segmentation

Below we:
- One-hot encode the enum features
- Sweep `k` and pick a good value (silhouette score)
- Generate cluster summaries + representative companies

Notes:
- `tokenMaturity` is currently constant (`high` for all rows), so it carries no information for clustering and is dropped.


In [39]:
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Drop constant columns for clustering
cluster_cols = [c for c in feature_cols if df[c].nunique() > 1]
cluster_cols

['productType',
 'collectionBucket',
 'uxMode',
 'imageryUsage',
 'colorStrategy',
 'typographyTone',
 'layoutDensity',
 'surfaceDepth',
 'themeMode',
 'shapeLanguage',
 'shadowStyle',
 'contentFocus',
 'motionUsage',
 'primaryIntent']

In [40]:
X = pd.get_dummies(df[cluster_cols])

# KMeans likes continuous space; standardize the one-hot matrix
Xs = StandardScaler(with_mean=False).fit_transform(X)

scores = []
for k in range(2, 11):
  km = KMeans(n_clusters=k, n_init='auto', random_state=7)
  labels = km.fit_predict(Xs)
  s = silhouette_score(Xs, labels)
  scores.append({'k': k, 'silhouette': s})

score_df = pd.DataFrame(scores).sort_values('silhouette', ascending=False)
score_df

,k,silhouette
0,2,0.142161
8,10,0.084099
7,9,0.079710
2,4,0.075418
6,8,0.073997
5,7,0.069142
4,6,0.059508
3,5,0.059004
1,3,0.051440


In [41]:
# Pick k (you can override this manually)
best_k = int(score_df.iloc[0]['k'])
best_k

2

In [42]:
km = KMeans(n_clusters=best_k, n_init='auto', random_state=7)
labels = km.fit_predict(Xs)

df_clustered = df.assign(cluster=labels)
df_clustered['cluster'].value_counts().sort_index()

cluster
0    47
1    11
Name: count, dtype: int64

In [43]:
# 2D projection for quick visual sanity check (not a "truth")
pca = PCA(n_components=2, random_state=7)
X2 = pca.fit_transform(Xs.toarray() if hasattr(Xs, 'toarray') else Xs)

plot_df = pd.DataFrame({
  'x': X2[:, 0],
  'y': X2[:, 1],
  'cluster': labels,
  'slug': df['slug'],
  'businessModel': df['collectionBucket'],
  'uxMode': df['uxMode']
})
plot_df.head()

,x,y,cluster,slug,businessModel,uxMode
0,5.171619,-3.583081,1,airbnb,enterpriseAndConsumer,browsingHeavy
1,-0.987879,-2.659232,0,airtable,designAndProductivity,browsingHeavy
2,5.199610,2.535619,1,apple,enterpriseAndConsumer,browsingHeavy
3,5.877857,1.887448,1,bmw,automotiveAndMobility,browsingHeavy
4,-2.173034,0.233778,0,cal,designAndProductivity,taskFocused


In [44]:
# Cluster summaries: dominant feature values + representative companies

def top_values(series: pd.Series, n: int = 3):
  vc = series.value_counts()
  total = vc.sum()
  return [f"{idx} ({cnt}/{total})" for idx, cnt in vc.head(n).items()]

summaries = []
for cid, sub in df_clustered.groupby('cluster'):
  row = {'cluster': cid, 'n': len(sub)}
  for c in cluster_cols:
    row[c] = '; '.join(top_values(sub[c], n=3))
  summaries.append(row)

summary_df = pd.DataFrame(summaries).sort_values('n', ascending=False)
summary_df

,cluster,n,productType,collectionBucket,uxMode,imageryUsage,colorStrategy,typographyTone,layoutDensity,surfaceDepth,themeMode,shapeLanguage,shadowStyle,contentFocus,motionUsage,primaryIntent
0,0,47,digital (47/47),developerToolsAndPlatforms (14/47); aiAndMachi...,browsingHeavy (14/47); mixed (14/47); taskFocu...,mixed (44/47); imageFirst (2/47); textFirst (1...,multiAccent (20/47); monochrome (11/47); singl...,technical (23/47); expressive (12/47); premium...,balanced (29/47); dense (14/47); sparse (4/47),flat (23/47); layered (18/47); subtle (6/47),lightFirst (25/47); darkFirst (16/47); dual (6...,pill (19/47); sharp (11/47); rounded (10/47),subtle (18/47); none (14/47); stacked (10/47),codeFirst (13/47); productScreenshots (12/47);...,low (25/47); medium (22/47),trust (37/47); emotionalBranding (5/47); explo...
1,1,11,physical (6/11); digital (4/11); marketplace (...,automotiveAndMobility (5/11); enterpriseAndCon...,browsingHeavy (11/11),imageFirst (11/11),singleAccent (7/11); multiAccent (2/11); gradi...,expressive (5/11); premium (2/11); technical (...,balanced (5/11); sparse (3/11); dense (3/11),flat (7/11); layered (4/11),darkFirst (5/11); lightFirst (3/11); dual (3/11),mixed (4/11); rounded (3/11); pill (2/11),none (6/11); stacked (3/11); subtle (2/11),photography (9/11); mixed (1/11); productScree...,medium (6/11); low (5/11),exploration (4/11); emotionalBranding (4/11); ...


In [45]:
def representatives(cluster_id: int, n: int = 8):
  sub = df_clustered[df_clustered['cluster'] == cluster_id]
  cols = ['slug', 'name', 'notes'] + cluster_cols
  return sub[cols].head(n)

representatives(int(summary_df.iloc[0]['cluster']), n=10)

,slug,name,notes,productType,collectionBucket,uxMode,imageryUsage,colorStrategy,typographyTone,layoutDensity,surfaceDepth,themeMode,shapeLanguage,shadowStyle,contentFocus,motionUsage,primaryIntent
1,airtable,Airtable,Airtable's design exudes a vibe of sophisticat...,digital,designAndProductivity,browsingHeavy,mixed,multiAccent,neutral,balanced,layered,lightFirst,rounded,subtle,productScreenshots,medium,trust
4,cal,Cal.com,"Cal.com embodies a clean, professional aesthet...",digital,designAndProductivity,taskFocused,mixed,monochrome,technical,balanced,layered,lightFirst,pill,stacked,productScreenshots,low,trust
5,claude,Claude,"Claude's design exudes a warm, intellectual vi...",digital,aiAndMachineLearning,browsingHeavy,mixed,multiAccent,expressive,balanced,layered,dual,rounded,ring,illustration,low,emotionalBranding
6,clay,Clay,"Clay's design exudes a warm, playful vibe, uti...",digital,designAndProductivity,mixed,mixed,multiAccent,expressive,balanced,layered,lightFirst,mixed,stacked,mixed,medium,emotionalBranding
7,clickhouse,ClickHouse,"ClickHouse's design exudes a high-performance,...",digital,infrastructureAndCloud,browsingHeavy,mixed,singleAccent,technical,dense,layered,darkFirst,sharp,subtle,codeFirst,low,trust
8,cohere,Cohere,"Cohere's design exudes a polished, enterprise-...",digital,aiAndMachineLearning,taskFocused,mixed,monochrome,technical,balanced,flat,lightFirst,mixed,none,productScreenshots,low,trust
9,coinbase,Coinbase,Coinbase's design exudes a clean and trustwort...,digital,fintechAndCrypto,browsingHeavy,mixed,singleAccent,neutral,balanced,flat,lightFirst,pill,none,productScreenshots,low,trust
10,composio,Composio,Composio's design exudes a developer-centric v...,digital,infrastructureAndCloud,taskFocused,mixed,monochrome,technical,dense,layered,darkFirst,sharp,subtle,codeFirst,low,trust
11,cursor,Cursor,"Cursor's design exudes a warm minimalism, util...",digital,developerToolsAndPlatforms,creationTool,mixed,multiAccent,premium,balanced,layered,lightFirst,pill,subtle,mixed,medium,trust
12,elevenlabs,ElevenLabs,The overall vibe of ElevenLabs' design is one ...,digital,aiAndMachineLearning,mixed,mixed,multiAccent,premium,balanced,layered,lightFirst,pill,stacked,mixed,low,emotionalBranding


In [46]:
# Save cluster assignments for later (can be imported to Neo4j too)
OUT_CLUSTER_CSV = REPO_ROOT / 'analysis' / 'clusters.csv'
df_clustered[['slug', 'cluster']].to_csv(OUT_CLUSTER_CSV, index=False)
print('Wrote', OUT_CLUSTER_CSV)

Wrote /Users/kihyvr/Documents/design-system-analysis/analysis/clusters.csv
